# Soup DPO: evidence on a free Colab T4
Select **Runtime → Change runtime type → T4 GPU** (Python 3.10–3.12), then Run all.
Upload `soup-dpo-takehome.zip` when asked. No paid services or secret tokens.
The run records errors and downloads a result ZIP even after failure.
Save the executed notebook with outputs too. A successful run does not imply SHIP.
The source ZIP is excluded from the returned archive; model cache is never exported.


In [ ]:
import os, sys, subprocess, zipfile
from pathlib import Path
from google.colab import files
assert (3, 10) <= sys.version_info[:2] <= (3, 12), 'Soup 0.75.2 supports Python 3.10–3.12; choose a compatible runtime version.'
uploaded = files.upload()
assert len(uploaded) == 1, 'Upload only soup-dpo-takehome.zip'
input_zip = next(iter(uploaded))
root = Path('/content/soup-dpo-takehome')
root.mkdir(exist_ok=True)
with zipfile.ZipFile(input_zip) as archive:
    for member in archive.infolist():
        destination = (root / member.filename).resolve()
        assert destination.is_relative_to(root.resolve()), 'Unsafe archive path'
    archive.extractall(root)
os.chdir(root)
assert Path('scripts/run_t4.py').exists()
print('Project:', root)


In [ ]:
# Record installation separately. The actual resolved stack is frozen by the runner.
install = subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requirements.txt'],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
Path('evidence').mkdir(exist_ok=True)
Path('evidence/colab-install.raw.log').write_bytes(install.stdout)
print(install.stdout.decode('utf-8', errors='replace'))
assert install.returncode == 0, 'Installation failed; preserve the raw log. Do not report training as successful.'


In [ ]:
# Each stage runs in a fresh process; no need to import the newly installed ML stack here.
# Streaming parity, lr=0 control and the 400-row DPO run all execute on this T4.
result = subprocess.run([sys.executable, '-m', 'scripts.run_t4'])
print('Pipeline exit code:', result.returncode)
print('All available logs and failure evidence are retained.')


In [ ]:
import json
latest_path = Path('evidence/latest_run.json')
if latest_path.exists():
    latest = json.loads(latest_path.read_text())
    print(latest)
    verdict = Path(latest['path']) / 'verdict.json'
    if verdict.exists():
        print(verdict.read_text())
print('Read reports/report.pdf and raw logs; do not infer learning from loss alone.')


In [ ]:
# Export even a failed attempt. Includes adapter + exact initialization when available.
export_path = '/content/soup-dpo-results.zip'
subprocess.run([sys.executable, '-m', 'scripts.package', '--include-weights', '--output', export_path], check=True)
files.download(export_path)


## Before submission
Download this notebook via **File → Download → Download .ipynb**, retaining outputs.
Keep every failed attempt. Extract `soup-dpo-results.zip` into the local project,
commit the evidence/report, and share the adapter archive privately with the reviewer.
Record your own review in `docs/AI_USAGE.md`; no personal review is pre-claimed.
The main report is exactly two pages. The final deployment call remains DON'T SHIP
for synthetic-only quality evidence, even if Soup's numeric ship gate passes.
